## Predict Wine Quality with Regularization

We’re looking at the red wine data in particular and while the original dataset has a 1-10 rating for each wine, we’ve made it a classification problem with a wine quality of good (>5 rating) or bad (<=5 rating). The goals of this project are to:

- implement different logistic regression classifiers
- find the best ridge-regularized classifier using hyperparameter tuning
- implement a tuned lasso-regularized feature selection method

What we’re working with:

11 input variables (based on physicochemical tests): ‘fixed acidity’, ‘volatile acidity’, ‘citric acid’, ‘residual sugar’,’chlorides’, ‘free sulfur dioxide’, ‘total sulfur dioxide’, ‘density’, ‘pH’, ‘sulphates’ and ‘alcohol’.
An output variable, ‘quality’ (0 for bad and 1 for good)

### Logistic Regression Classifier without Regularization

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# import data
df = pd.read_csv('wine_quality.csv') # reads the wine quality data into a DataFrame
print(df.columns) # prints the column names
y = df['quality'] # sets y as the target (quality),
features = df.drop(columns = ['quality']) # creates features by dropping the quality column

# 1. Data transformation
from sklearn.preprocessing import StandardScaler
standard_scaler_fit = StandardScaler().fit(features) #fits a StandardScaler to your features
X = standard_scaler_fit.transform(features) # transforms the features so each has mean 0 and standard deviation 1. (This helps many machine learning models perform better)

## 2. Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 99) # splits your data into training and test sets
# - X_train and y_train are used to train your model,
# - X_test and y_test help you check how well your model performs.

## 3. Fit a logistic regression classifier without regularization
from sklearn.linear_model import LogisticRegression
clf_no_reg = LogisticRegression(penalty = 'none') #creates a classifier with no regularization with penalty = 'none'
clf_no_reg.fit(X_train, y_train) # fit to training data to train model

## 4. Plot the coefficients
predictors = features.columns
coefficients = clf_no_reg.coef_.ravel()
coef = pd.Series(coefficients,predictors).sort_values()
coef.plot(kind='bar', title = 'Coefficients (no regularization)')
plt.tight_layout()
plt.show()
plt.clf()
```
![No regularization](images/no_regularization_plot.png)

```python
## 5. Training and test performance
from sklearn.metrics import f1_score
y_pred_test = clf_no_reg.predict(X_test)
y_pred_train = clf_no_reg.predict(X_train)
print('Training Score', f1_score(y_train, y_pred_train))
print('Testing Score', f1_score(y_test, y_pred_test))
# Training Score 0.7727598566308242
# Testing Score 0.7266666666666667
```

### Logistic Regression with L2 Regularization

```python
## 6. Default Implementation (L2-regularized!)
clf_default = LogisticRegression() # initialize Logistic regression model, default is L2 regularization
clf_default.fit(X_train, y_train) # fit training data to model

## 7. Ridge Scores
y_pred_train = clf_default.predict(X_train)
y_pred_test = clf_default.predict(X_test)
print('Ridge-regularized Training Score', f1_score(y_train, y_pred_train))
print('Ridge-regularized Testing Score', f1_score(y_test, y_pred_test))
# Ridge-regularized Training Score 0.7727598566308242
# Ridge-regularized Testing Score 0.7266666666666667

## 8. Coarse-grained hyperparameter tuning
training_array = []
test_array = []
C_array = [0.0001, 0.001, 0.01, 0.1, 1]

for x in C_array: # loop through array C value, apply to C to obtain optimal C value
  clf = LogisticRegression(C = x)
  clf.fit(X_train, y_train)
  y_pred_train = clf.predict(X_train)
  training_array.append(f1_score(y_train, y_pred_train))
  y_pred_test = clf.predict(X_test)
  test_array.append(f1_score(y_test, y_pred_test))

print('Training: ', training_array)
print('Test: ', test_array)
# Training:  [0.7074279939363315, 0.763140817650876, 0.76802284082798, 0.7713261648745521, 0.7727598566308242]
# Test:  [0.6526315789473685, 0.7407407407407408, 0.7302631578947368, 0.7350993377483445, 0.7266666666666667]

## 9. Plot training and test scores as a function of C
plt.plot(C_array,training_array)
plt.plot(C_array,test_array)
plt.xscale('log')
plt.show()
plt.clf()
```

![C Graph](images/C_graph.png)

### Hyperparameter Tuning for L2 Regularization

```python
## 10. Making a parameter grid for GridSearchCV
np.logspace(-4, -2, 100)
tuning_C = {'C':C_array} # dict to enter as an input to param_grid

## 11. Implementing GridSearchCV with l2 penalty
from sklearn.model_selection import GridSearchCV
clf_gs = LogisticRegression()
gs = GridSearchCV(clf_gs, param_grid = tuning_C, scoring = 'f1', cv=5)
gs.fit(X_train,y_train)

## 12. Optimal C value and the score corresponding to it
print('Parameter:', gs.best_params_, 'Score:', gs.best_score_)
# Parameter: {'C': 0.1} Score: 0.7701167945675984

## 13. Validating the "best classifier"
clf_best_ridge = LogisticRegression(C = gs.best_params_['C'])
clf_best_ridge.fit(X_train, y_train)
y_pred_best = clf_best_ridge.predict(X_test)
print('Best Ridge Test Score:', f1_score(y_test, y_pred_best))
# Best Ridge Test Score: 0.7350993377483445
```

### Feature Selection using L1 Regularization

```python
## 14. Implement L1 hyperparameter tuning with LogisticRegressionCV
from sklearn.linear_model import LogisticRegressionCV
C_array = np.logspace(-2, 2, 100)
clf_l1 = LogisticRegressionCV(Cs = C_array, cv=5, penalty='l1', solver='liblinear', scoring='f1')
clf_l1.fit(X,y)

## 15. Optimal C value and corresponding coefficients
print('Optimal C value', clf_l1.C_, 'Coefficients', clf_l1.coef_)
# Optimal C value [0.25950242] Coefficients [[ 0.11598177  -0.55610136  -0.17225269  0.03124289  -0.17343099  0.20068889  -0.50609606  0.  -0.07065245  0.43227124  0.94096873]]

## 16. Plotting the tuned L1 coefficients
coefficients = clf_l1.coef_.ravel()
coef = pd.Series(coefficients,predictors).sort_values()

plt.figure(figsize = (12,8))
coef.plot(kind='bar', title = 'Coefficients for tuned L1')
plt.tight_layout()
plt.show()
plt.clf()
```

![L1 Regularization](images/l1_regularization_plot.png)